# Data Pipelines for Machine Learning: Streaming, Windows, Deduplication & Backfilling

High-throughput ML training and real-time inference rely on robust data engineering backbones:
1. **Idempotence & Deduplication**: Ensuring at-least-once message queues don't corrupt feature aggregations with duplicates.
2. **Time-Windowed Stream Processing**: Computing tumbling and sliding aggregations (e.g., Flink / Spark Streaming patterns).
3. **Dead-Letter Queues (DLQ)**: Isolating poisoned or malformed payloads without crashing the ingestion pipeline.
4. **Historical Data Backfill**: Partitioning historical intervals for parallel reconstruction of training datasets.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('08-system-design/data-pipelines/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import pipeline_core
print('pipeline_core loaded')

## 1. Idempotent Deduplication
Filtering out duplicate events caused by network retries in Kafka/SQS streams.

In [ ]:
from pipeline_core import IdempotentDeduplicator

dedup = IdempotentDeduplicator(max_history=100)

incoming_events = [
    ("evt_101", "User clicked item A"),
    ("evt_102", "User added item A to cart"),
    ("evt_101", "User clicked item A (DUPLICATE RETRY)"),
    ("evt_103", "User purchased item A")
]

for event_id, description in incoming_events:
    is_dup = dedup.is_duplicate_and_record(event_id)
    status = "IGNORED (DUPLICATE)" if is_dup else "PROCESSED"
    print(f"[{status:20s}] Event ID={event_id}: {description}")

## 2. Sliding Window Real-Time Feature Aggregator
Calculating continuous moving statistics (e.g., user transaction count in the last 60 seconds).

In [ ]:
from pipeline_core import SlidingWindowAggregator

# 60-second sliding feature aggregator (e.g. transaction amounts)
window_agg = SlidingWindowAggregator(window_size_seconds=60.0)

# Stream of events: (timestamp in seconds, amount in dollars)
stream_events = [
    (10.0, 150.0),
    (25.0, 200.0),
    (50.0, 350.0)
]
for ts, amt in stream_events:
    window_agg.add_event(ts, amt)

# Query window at t=60s
res_60 = window_agg.compute_window(current_time=60.0)
print(f"Window at t=60s: Count={res_60.count}, Sum=${res_60.sum_value:.2f}, Mean=${res_60.mean_value:.2f}")

# Event arrives at t=80s
window_agg.add_event(80.0, 500.0)

# Query window at t=90s (window covers [30s, 90s] -> drops events from t=10s and t=25s)
res_90 = window_agg.compute_window(current_time=90.0)
print(f"Window at t=90s: Count={res_90.count}, Sum=${res_90.sum_value:.2f}, Mean=${res_90.mean_value:.2f}")

## 3. Dead-Letter Queue (DLQ) for Poisonous Payloads
Isolating malformed data so healthy records proceed uninterrupted.

In [ ]:
from pipeline_core import DeadLetterQueue

dlq = DeadLetterQueue()

def parse_record(payload):
    if "user_id" not in payload:
        raise KeyError("Missing required field: 'user_id'")
    return f"Valid user {payload['user_id']}"

test_records = [
    {"user_id": "u1", "amount": 100},
    {"corrupted_field": True},
    {"user_id": "u2", "amount": 250}
]

for rec in test_records:
    try:
        msg = parse_record(rec)
        print("Success:", msg)
    except Exception as e:
        dlq.push_failure(rec, e)
        print(f"Failed record routed to DLQ! Reason: {e}")

print(f"\nDLQ current size: {dlq.size()} failed records stored for review.")

## 4. Parallel Backfill Range Partitioning
Slicing long historical intervals into non-overlapping batches for distributed Spark/Ray jobs.

In [ ]:
from pipeline_core import partition_backfill_ranges

# 12-hour backfill window partitioned into 2-hour worker chunks
chunks = partition_backfill_ranges(start_time=0.0, end_time=43200.0, chunk_size_seconds=7200.0)
print(f"Split 12 hours into {len(chunks)} parallel worker tasks:")
for i, (start, end) in enumerate(chunks, 1):
    print(f" - Worker Task {i}: Range [{start/3600:.1f}h -> {end/3600:.1f}h]")